# RescueVision v2 - joint model eval rerun (hardened)

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
mark("setup:start")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "albumentations==2.0.8"], check=False)
import albumentations as _A
mark("setup:albumentations", extra=_A.__version__)
print("albumentations", _A.__version__)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"], check=False)
import segmentation_models_pytorch as smp
print("smp", smp.__version__)
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NW = 0
PIN = False
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
import numpy as np
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

from rescuevision.constants import IMAGENET_MEAN, IMAGENET_STD
from rescuevision.data import datasets as _D
from PIL import Image as _PIL
from pathlib import Path as _PPath

def _load_arr(path):
    path = _PPath(path)
    if path.suffix == ".npy":
        return np.load(path)
    return np.array(_PIL.open(path))

def _find_tile(directory, tile_id):
    for ext in (".png", ".npy"):
        p = _PPath(directory) / f"{tile_id}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"{tile_id} in {directory}")

def _tile_dataset_init(self, processed_dir, split_csv, split="train", mask_type="building_binary",
                       transform=None, image_size=512, debug_limit=None):
    import pandas as pd
    self.processed_dir = _PPath(processed_dir)
    self.images_dir = self.processed_dir / "images" / "pre"
    self.pre_dir = self.processed_dir / "images" / "pre"
    self.post_dir = self.processed_dir / "images" / "post"
    if type(self).__name__ == "XBDDamageSegmentationDataset":
        self.masks_dir = self.processed_dir / "masks" / "damage_multiclass"
    elif type(self).__name__ == "XBDChangeDetectionDataset":
        self.masks_dir = self.processed_dir / "masks" / "change_binary"
    else:
        self.masks_dir = self.processed_dir / "masks" / mask_type
    self.transform = transform
    self.image_size = image_size
    splits_df = pd.read_csv(split_csv)
    tile_ids = splits_df[splits_df["split"] == split]["tile_id"].tolist()
    if debug_limit:
        tile_ids = tile_ids[:debug_limit]
    self.tile_ids = [t for t in tile_ids
                     if _find_tile(self.pre_dir, t).exists()
                     and _find_tile(self.post_dir, t).exists()
                     and _find_tile(self.masks_dir, t).exists()]
    print(f"{type(self).__name__}: split={split}, {len(self.tile_ids)} tiles")

def _xbd_seg_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    image = _load_arr(_find_tile(self.pre_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    if self.transform:
        aug = self.transform(image=image, mask=mask)
        image, mask = aug["image"], aug["mask"]
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"image": image, "mask": mask, "tile_id": tile_id}

_D.XBDSegmentationDataset.__init__ = _tile_dataset_init
_D.XBDSegmentationDataset.__getitem__ = _xbd_seg_getitem
mark("patch:done")
print("png-aware dataset patch applied")

In [ ]:
import time
import pandas as pd
mark("tar:wait")
TAR = None
if (Path("/tmp/processed_xbd") / "metadata" / "splits_event_aware.csv").exists():
    print("processed tree already at /tmp/processed_xbd (smoke/debug path)")
else:
    for _attempt in range(40):
        for _p in Path("/kaggle/input").rglob("processed_xbd_native.tar"):
            TAR = _p
            break
        if TAR is not None:
            break
        _ents = [p.name for p in Path("/kaggle/input").iterdir()] if Path("/kaggle/input").exists() else []
        print(f"[{_attempt+1}/40] waiting for tar mount, /kaggle/input: {_ents}", flush=True)
        time.sleep(30)
    assert TAR is not None, "processed_xbd_native.tar never mounted - kernel-source attach failed"
    _gib = TAR.stat().st_size / 1e9
    print("tar:", TAR, f"{_gib:.2f} GB")
    mark("tar:mounted", extra=f"{_gib:.2f}GB")
    subprocess.run(["tar", "-xf", str(TAR), "-C", "/tmp"], check=True)
PROCESSED = Path("/tmp/processed_xbd")
assert (PROCESSED / "metadata" / "splits_event_aware.csv").exists(), "extracted tree missing splits csv"
_splits = pd.read_csv(PROCESSED / "metadata" / "splits_event_aware.csv")
print("tiles per split:", _splits["split"].value_counts().to_dict())
_tiles_meta = pd.read_csv(PROCESSED / "metadata" / "tiles.csv")
print("shift columns:", [c for c in _tiles_meta.columns if "shift" in c])
mark("tar:extracted", extra=str(_splits["split"].value_counts().to_dict()))

In [ ]:
# ---- shared eval kit: threshold sweep + approximate official xView2 score ----
import numpy as np
from scipy.ndimage import label as _cc_label

THRS = np.arange(0.20, 0.81, 0.05)

def sweep_loc_thresholds(probs, gts):
    """pooled inter/union per threshold -> {thr: IoU}"""
    out = {}
    p = np.concatenate(probs); g = np.concatenate(gts).astype(bool)
    for t in THRS:
        pred = p > t
        inter = np.logical_and(pred, g).sum()
        union = np.logical_or(pred, g).sum()
        out[round(float(t), 2)] = float(inter) / max(union, 1)
    return out

def best_thr(probs, gts):
    pooled = sweep_loc_thresholds(probs, gts)
    thr = max(pooled, key=pooled.get)
    return float(thr), pooled[thr], pooled

def _xview2_tile(loc_pred, loc_gt, dmg_pred, dmg_gt):
    """greedy component match: returns (fp, fn, matches=[(iou, pred_grade, gt_grade)])"""
    p_lab, np_ = _cc_label(loc_pred)
    g_lab, ng_ = _cc_label(loc_gt)
    fp = 0
    matches = []
    used = np.zeros(ng_ + 1, dtype=bool)
    if np_ and ng_:
        if np_ > 3000 or ng_ > 3000:
            return int(np_), int(ng_), []   # pathological blob count - skip matching
        pair = p_lab.astype(np.int32) * np.int32(ng_ + 1) + g_lab.astype(np.int32)
        hist = np.bincount(pair.ravel(), minlength=(np_ + 1) * (ng_ + 1)).reshape(np_ + 1, ng_ + 1)
        areas_p = hist[1:].sum(axis=1); areas_g = hist[:, 1:].sum(axis=0)
        for pi in range(np_):
            inters = hist[pi + 1, 1:]
            gi = int(np.argmax(inters))
            inter = int(inters[gi])
            if inter == 0:
                fp += 1
                continue
            union = int(areas_p[pi] + areas_g[gi] - inter)
            iou = inter / union
            if iou >= 0.5 and not used[gi]:
                used[gi] = True
                m = hist[pi + 1, gi + 1] > 0
                pg = dmg_pred[m]; gg = dmg_gt[m]
                pg = pg[pg > 0]; gg = gg[gg > 0]
                pgr = int(np.bincount(pg).argmax()) if len(pg) else 0
                ggr = int(np.bincount(gg).argmax()) if len(gg) else 0
                matches.append((iou, pgr, ggr))
            else:
                fp += 1
    else:
        fp += int(np_)
    fn = int((~used[1:]).sum())
    return fp, fn, matches

def xview2_score(tiles, thr):
    """tiles: [{loc_prob,loc_gt,dmg_pred,dmg_gt}] -> approximate official xView2 score."""
    stats = {t: [0, 0, 0] for t in (0.5, 0.6, 0.7, 0.8)}   # tp, fp, fn
    matches50 = []
    for d in tiles:
        pred = d["loc_prob"] > thr
        fp, fn, matches = _xview2_tile(pred, d["loc_gt"], d["dmg_pred"], d["dmg_gt"])
        for t in stats:
            tp_t = sum(1 for iou, _, _ in matches if iou >= t)
            fp_t = fp + sum(1 for iou, _, _ in matches if iou < t)
            fn_t = fn + sum(1 for iou, _, _ in matches if iou < t)
            stats[t][0] += tp_t; stats[t][1] += fp_t; stats[t][2] += fn_t
        matches50 += matches
    _f1 = lambda tp, fp, fn: 2 * tp / max(2 * tp + fp + fn, 1)
    loc_f1 = float(np.mean([_f1(*stats[t]) for t in stats]))
    per_class = {}
    for c in (1, 2, 3, 4):
        tp_c = sum(1 for _, p, g in matches50 if p == c and g == c)
        fp_c = sum(1 for _, p, g in matches50 if p == c and g != c)
        fn_c = sum(1 for _, p, g in matches50 if p != c and g == c)
        per_class[c] = round(2 * tp_c / max(2 * tp_c + fp_c + fn_c, 1), 4)
    dmg_f1 = float(np.mean(list(per_class.values())))
    return {"loc_f1": round(loc_f1, 4), "damage_f1": round(dmg_f1, 4),
            "overall_f1": round(0.3 * loc_f1 + 0.7 * dmg_f1, 4),
            "per_class_damage_f1": per_class,
            "n_pred_buildings": int(sum(s[0] + s[1] for s in stats.values()) // 8)}
print("eval kit ready")

In [ ]:
import json, math
import numpy as np
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader

name = "joint_damage_siamese_v2"
SIZE = 608
BS = 8

def phase(label, fn):
    try:
        r = fn()
        mark(f"eval:{label}:ok", extra=str(r)[:300])
        return r
    except Exception:
        mark(f"eval:{label}:FAIL", extra=traceback.format_exc()[-1400:])
        raise

import torch as _torch
import torch.nn as _nn
import torch.nn.functional as _F
import segmentation_models_pytorch as _smp


class SiameseUNet(_nn.Module):
    """Shared-encoder siamese U-Net with fused pre/post features (fixed decoder)."""

    def __init__(self, encoder_name="resnet34", encoder_weights="imagenet",
                 fusion_mode="concat_absdiff", classes=1):
        super().__init__()
        self.fusion_mode = fusion_mode
        self.classes = classes

        base_unet = _smp.Unet(encoder_name=encoder_name, encoder_weights=encoder_weights,
                              in_channels=3, classes=classes)
        self.encoder = base_unet.encoder
        encoder_channels = self._get_encoder_channels()
        bottleneck_ch = encoder_channels[-1]

        if fusion_mode == "concat":
            fusion_mult = 2
        elif fusion_mode == "absdiff":
            fusion_mult = 1
        else:
            fusion_mult = 3   # concat_absdiff

        skip_channels = list(reversed(encoder_channels[:-1]))
        decoder_channels = [256, 128, 64, 32]

        self.skip_convs = _nn.ModuleList()
        for enc_ch in skip_channels:
            self.skip_convs.append(_nn.Sequential(
                _nn.Conv2d(enc_ch * fusion_mult, enc_ch, 1, bias=False),
                _nn.BatchNorm2d(enc_ch),
                _nn.ReLU(inplace=True),
            ))

        self.decoder = _nn.ModuleList()
        in_ch = bottleneck_ch * fusion_mult + skip_channels[0]
        for i, dec_ch in enumerate(decoder_channels):
            self.decoder.append(_nn.Sequential(
                _nn.Conv2d(in_ch, dec_ch, 3, padding=1, bias=False),
                _nn.BatchNorm2d(dec_ch),
                _nn.ReLU(inplace=True),
                _nn.Conv2d(dec_ch, dec_ch, 3, padding=1, bias=False),
                _nn.BatchNorm2d(dec_ch),
                _nn.ReLU(inplace=True),
            ))
            if i + 1 < len(decoder_channels):
                in_ch = dec_ch + skip_channels[i + 1]

        self.segmentation_head = _nn.Conv2d(decoder_channels[-1], classes, 1)

    def _get_encoder_channels(self):
        channels = []
        dummy = _torch.randn(1, 3, 64, 64)
        with _torch.no_grad():
            features = self.encoder(dummy)
        return [f.shape[1] for f in features]

    def _fuse(self, a, b):
        if self.fusion_mode == "concat":
            return _torch.cat([a, b], dim=1)
        if self.fusion_mode == "absdiff":
            return _torch.abs(b - a)
        return _torch.cat([a, b, _torch.abs(b - a)], dim=1)

    def forward(self, pre_image, post_image):
        features_pre = self.encoder(pre_image)
        features_post = self.encoder(post_image)
        x = self._fuse(features_pre[-1], features_post[-1])
        for i, (dec_block, skip_conv) in enumerate(zip(self.decoder, self.skip_convs)):
            x = _F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            skip = skip_conv(self._fuse(features_pre[-(i + 2)], features_post[-(i + 2)]))
            if x.shape[2:] != skip.shape[2:]:
                x = _F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
            x = dec_block(_torch.cat([x, skip], dim=1))
        if x.shape[2:] != pre_image.shape[2:]:
            x = _F.interpolate(x, size=pre_image.shape[2:], mode="bilinear", align_corners=False)
        return self.segmentation_head(x)


print("SiameseUNet (fixed decoder) defined")

# 1. checkpoint (kernel-source attach is impossible for an ERRORED version, so
#    download it from the signed output URL; rglob fallback covers the future
#    case where a successful run's output IS mounted)
BESTPT_URL = "https://www.kaggleusercontent.com/kf/358046840/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..idPtduVD-Jw3mvUqc1UI9A.96ihL3-dQV_GwUUD9R7j1dIuv95b7SdWgPiMKkptj08KCkJ57qhRdwaUjffap0JDHlb9Tud88OPASqjHac4kmYSKCtX9X-EsRSrUnRRqdtblddF3m4v89l1chHIRWyuwqgp69uZtb6GF-vmn81OSY5G7pQyhkEx-vjmLOI6NaiwxGWgDpWtsHTZRl6WFWR_iJzplba8kj_baMZqYt-com9P1OG2KlY2CVCYF_P-IgefLAZYsIOAxaePjD7Xz2PBC27V5zI1sm5Z4elEanm7Nsv3nwMwx02CI0NIoFHlygbLAUqpVaf0-1Ky3-qNsnMCMMtPFX0ipHsbT9RUz0AaJ7NSgpAazzpmCwq-wVotMT0tu8qViP3hs1qMjOQYIlU-pviAJcUsoDiqlaqdOKIJIE5Amgju5JXRsqltx8OgCf4tfJTUwZl1yjC48B3UbH-d_czwPf8G6WgxzNXXJDkfaeGmCyoZ4eFVpJEFvIjQrXx5FDb8curwfbLyoySdwZM5i3nz_uPwjLx-vMkJh62sRi-1vet3OWSlPGlPO6q420Q1xJWZla7BCsTUPACnc32ripePESnpij5cxRZQQQo7VgcEywKqDmMnpjVMRv0jtD19KFiXERePoAQA-4hLZExga.erHHletXK5OLZbnd-ZGMcQ/joint_damage_siamese_v2/checkpoints/best.pt"
assert BESTPT_URL.startswith("https://"), "missing signed URL"

def _find_ckpt():
    for p in Path("/kaggle/input").rglob("best.pt"):
        return p
    dest = "/tmp/joint_best.pt"
    subprocess.run(["curl", "-sL", "--retry", "3", "-o", dest, BESTPT_URL], check=True)
    sz = os.path.getsize(dest)
    assert sz > 10_000_000, f"download too small: {sz}"
    print(f"downloaded checkpoint: {sz/1e6:.1f} MB")
    return Path(dest)
CKPT_PATH = phase("find-ckpt", _find_ckpt)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
model = SiameseUNet(encoder_name="resnet34", encoder_weights=None,
                    fusion_mode="concat_absdiff", classes=5).to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("ckpt:", CKPT_PATH, "epoch", ckpt["epoch"], "combined", ckpt.get("combined"))

class JointDamageDataset(Dataset):
    def __init__(self, root, split_csv, split="train", transform=None):
        self.root = Path(root)
        df = pd.read_csv(split_csv)
        ids = df[df["split"] == split]["tile_id"].tolist()
        self.tile_ids = [t for t in ids
                         if (self.root / "images/pre" / f"{t}.png").exists()
                         and (self.root / "masks/damage_multiclass_dil" / f"{t}.png").exists()]
        self.transform = transform
        print(f"JointDamageDataset split={split}: {len(self.tile_ids)} tiles")

    def __len__(self):
        return len(self.tile_ids)

    def __getitem__(self, idx):
        tid = self.tile_ids[idx]
        pre = _load_arr(self.root / "images/pre" / f"{tid}.png")
        post = _load_arr(self.root / "images/post" / f"{tid}.png")
        dmg = _load_arr(self.root / "masks/damage_multiclass_dil" / f"{tid}.png").astype(np.uint8)
        loc = _load_arr(self.root / "masks/building_binary" / f"{tid}.png").astype(np.uint8)
        comb = np.where((loc > 0) & (dmg == 0), 6, dmg).astype(np.uint8)
        pair = np.concatenate([pre, post], axis=2)
        if self.transform:
            aug = self.transform(image=pair, mask=comb)
            pair_t, comb_t = aug["image"], aug["mask"]
        else:
            pair_t = torch.from_numpy(pair.transpose(2, 0, 1))
            comb_t = torch.from_numpy(comb)
        comb_t = comb_t.long() if torch.is_tensor(comb_t) else torch.from_numpy(np.asarray(comb_t)).long()
        loc_t = (comb_t > 0).float()
        dmg_t = comb_t.clone() - 1
        dmg_t[(comb_t < 1) | (comb_t > 4)] = 255
        return {"pre": pair_t[:3].float(), "post": pair_t[3:].float(),
                "loc": loc_t, "dmg": dmg_t, "tile_id": tid}

def joint_pair_transforms(phase_name="train", size=608):
    mean, std = IMAGENET_MEAN * 2, IMAGENET_STD * 2
    tf = [A.Resize(size, size)]
    tf += [A.Normalize(mean=mean, std=std), ToTensorV2()]
    return A.Compose(tf, is_check_shapes=False)

@torch.no_grad()
def tta_joint(pre, post):
    outs = []
    for k in range(4):
        a, b = pre, post
        if k == 1: a, b = torch.flip(pre, [3]), torch.flip(post, [3])
        elif k == 2: a, b = torch.flip(pre, [2]), torch.flip(post, [2])
        elif k == 3: a, b = torch.flip(pre, [2, 3]), torch.flip(post, [2, 3])
        o = model(a, b)
        if k == 1: o = torch.flip(o, [3])
        elif k == 2: o = torch.flip(o, [2])
        elif k == 3: o = torch.flip(o, [2, 3])
        outs.append(o)
    o = torch.stack(outs, 0).mean(0).float()
    return torch.sigmoid(o[:, 0]), torch.softmax(o[:, 1:], 1)

def collect(split, limit, thr):
    ds = JointDamageDataset(PROCESSED, split_csv, split, joint_pair_transforms("val", SIZE))
    ds.tile_ids = ds.tile_ids[:limit]
    loader = DataLoader(ds, batch_size=BS, shuffle=False, num_workers=NW)
    probs, gts, tiles = [], [], []
    tp = np.zeros(4); fp = np.zeros(4); fn = np.zeros(4)
    inter = union = 0
    for b in loader:
        with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type == "cuda")):
            loc_p, dmg_p = tta_joint(b["pre"].to(DEVICE), b["post"].to(DEVICE))
        loc_p = loc_p.float().cpu().numpy(); dmg_p = dmg_p.float().cpu().numpy()
        loc_t = b["loc"].numpy() > 0.5
        dmg_t = b["dmg"].numpy()
        pred_bin = loc_p > thr
        inter += np.logical_and(pred_bin, loc_t).sum(); union += np.logical_or(pred_bin, loc_t).sum()
        probs.append(loc_p); gts.append(loc_t)
        for i in range(loc_p.shape[0]):
            pred_g = np.where(pred_bin[i], np.argmax(dmg_p[i], 0) + 1, 0)
            gt_g = np.where(loc_t[i], np.where(dmg_t[i] != 255, dmg_t[i] + 1, 1), 0)
            tiles.append({"loc_prob": loc_p[i].astype(np.float32), "loc_gt": loc_t[i],
                          "dmg_pred": pred_g.astype(np.int16), "dmg_gt": gt_g.astype(np.int16)})
        valid = dmg_t != 255
        if valid.sum():
            pred = dmg_p.argmax(1)
            for c in range(4):
                p = (pred == c) & valid; g = (dmg_t == c) & valid
                tp[c] += (p & g).sum(); fp[c] += (p & ~g).sum(); fn[c] += (g & ~p).sum()
    pix_iou = inter / max(union, 1)
    f1s = [2 * tp[c] / max(2 * tp[c] + fp[c] + fn[c], 1) for c in range(4)]
    return probs, gts, tiles, pix_iou, f1s

split_csv = PROCESSED / "metadata" / "splits_event_aware.csv"

# 2. threshold sweep on val subset
def _sweep():
    p, g, _, _, _ = collect("val", 250, 0.5)
    pooled = sweep_loc_thresholds(p, g)
    thr = max(pooled, key=pooled.get)
    return float(thr), pooled[thr], pooled, p, g
thr, thr_iou, pooled, _, _ = phase("sweep", _sweep)
print(f"thr sweep: best={thr:.2f} IoU={thr_iou:.4f}")

# 3. test metrics + approximate xView2
def _test():
    return collect("test", 300, thr)
p2, g2, tiles, pix_iou, f1s = phase("test-collect", _test)
print("TEST pixel loc IoU @thr:", round(float(pix_iou), 4),
      "dmg F1:", [round(float(f), 4) for f in f1s])
xv = phase("xview2", lambda: xview2_score(tiles, thr))
print("APPROX xView2:", json.dumps(xv))

# 4. summary + plot
def _summary():
    summary = {"run": name, "best_epoch": int(ckpt["epoch"]),
               "val_combined": round(float(ckpt["combined"]), 4),
               "val": {"loc_iou": round(float(ckpt["val"]["loc_iou"]), 4),
                       "dmg_macro_f1": round(float(ckpt["val"]["dmg_macro_f1"]), 4),
                       "class_f1": [round(float(x), 4) for x in ckpt["val"]["class_f1"]]},
               "threshold": float(thr), "thr_sweep_val_iou": round(float(thr_iou), 4),
               "test": {"pixel_loc_iou": round(float(pix_iou), 4),
                        "pixel_dmg_f1": [round(float(f), 4) for f in f1s],
                        "pixel_dmg_macro_f1": round(float(np.mean(f1s)), 4)},
               "xview2_approx": xv, "n_test_tiles": len(tiles)}
    out = WORK / name
    out.mkdir(parents=True, exist_ok=True)
    with open(out / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    d0 = tiles[0]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(d0["loc_prob"], vmin=0, vmax=1); axes[0].set_title("loc prob")
    axes[1].imshow(d0["dmg_gt"], vmin=0, vmax=4, cmap="turbo"); axes[1].set_title("GT grade")
    axes[2].imshow(d0["dmg_pred"], vmin=0, vmax=4, cmap="turbo"); axes[2].set_title("pred grade")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.savefig(out / "test_predictions.png", dpi=120, bbox_inches="tight")
    plt.close("all")
    return summary
summary = phase("summary", _summary)
print(json.dumps(summary, indent=2))
mark("eval:ALL-DONE", extra=str(xv))